In [1]:
import pandas as pd
from pathlib import Path

In [2]:

file_path = Path(
    "../Data/2020/2020-06/2020-06-01/2020-06-01-TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93.parquet"
)

df = pd.read_parquet(file_path)

In [3]:
df.shape

(696160, 17)

In [4]:
print(df.columns.tolist())

['player_name', 'time', 'lat', 'lon', 'speed', 'heart_rate', 'hacc', 'hdop', 'signal_quality', 'num_satellites', 'inst_acc_impulse', 'accl_x', 'accl_y', 'accl_z', 'gyro_x', 'gyro_y', 'gyro_z']


In [5]:
df.head()

,player_name,time,lat,lon,speed,heart_rate,hacc,hdop,signal_quality,num_satellites,inst_acc_impulse,accl_x,accl_y,accl_z,gyro_x,gyro_y,gyro_z
0,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,13:46:01.8,63.444867,10.451694,0.0,0,0,3,345,22,0.0,0.282031,0.869243,0.700000,-0.006348,-0.059326,0.033691
1,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,13:46:01.8,63.444867,10.451694,0.0,0,0,3,345,22,0.0,0.273438,0.866776,0.701562,-0.004150,-0.056152,0.030029
2,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,13:46:01.8,63.444867,10.451694,0.0,0,0,3,345,22,0.0,0.273438,0.865132,0.702344,-0.002930,-0.055176,0.025391
3,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,13:46:01.8,63.444867,10.451694,0.0,0,0,3,345,22,0.0,0.278906,0.865132,0.705469,-0.002441,-0.054199,0.020752
4,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,13:46:01.8,63.444867,10.451694,0.0,0,0,3,345,22,0.0,0.284375,0.859375,0.711719,-0.001465,-0.054199,0.016602


# Creating Summary Session

In [6]:
session_summary = {
    "player_name": df["player_name"].iloc[0],

    "avg_speed": df["speed"].mean(),

    "max_speed": df["speed"].max(),

    "avg_heart_rate": df["heart_rate"].mean(),

    "max_heart_rate": df["heart_rate"].max(),

    "avg_accel_x": df["accl_x"].mean(),

    "avg_accel_y": df["accl_y"].mean(),

    "avg_accel_z": df["accl_z"].mean(),

    "rows": len(df)
}

session_summary

# What we did is we transformed the raw GPS data into a summary of the session, which includes average and maximum speed, heart rate, and acceleration in each direction. This summary can be used for further analysis or to compare different sessions.
"""Why Is This Important?
Machine Learning cannot use:
696,160 rows per session
efficiently.
Instead we create:"""

'Why Is This Important?\nMachine Learning cannot use:\n696,160 rows per session\nefficiently.\nInstead we create:'

# Extracting distance covered per quarter per running type (high speed, low speed)

In [7]:
df["time"].head()

0    13:46:01.8
1    13:46:01.8
2    13:46:01.8
3    13:46:01.8
4    13:46:01.8
Name: time, dtype: str

In [8]:
#Convert Time Column
df["time"] = pd.to_timedelta(df["time"])

In [9]:
#Create Session Progress
session_start = df["time"].min()

df["elapsed_seconds"] = (
    df["time"] - session_start
).dt.total_seconds()

df[["time", "elapsed_seconds"]].head()

,time,elapsed_seconds
0,0 days 13:46:01.800000,0.0
1,0 days 13:46:01.800000,0.0
2,0 days 13:46:01.800000,0.0
3,0 days 13:46:01.800000,0.0
4,0 days 13:46:01.800000,0.0


In [10]:
#Create Quarters
session_duration = df["elapsed_seconds"].max()

quarter_length = session_duration / 4

print("Session Duration:", session_duration)
print("Quarter Length:", quarter_length)

Session Duration: 6962.2
Quarter Length: 1740.55


In [11]:
#Create quarter labels:
df["quarter"] = pd.cut(
    df["elapsed_seconds"],
    bins=[
        0,
        quarter_length,
        quarter_length * 2,
        quarter_length * 3,
        session_duration
    ],
    labels=["Q1", "Q2", "Q3", "Q4"],
    include_lowest=True
)

df[["elapsed_seconds", "quarter"]].tail()

,elapsed_seconds,quarter
696155,6962.2,Q4
696156,6962.2,Q4
696157,6962.2,Q4
696158,6962.2,Q4
696159,6962.2,Q4


In [12]:
#Step 2.4 — Define Running Types
df["running_type"] = df["speed"].apply(
    lambda x: "high_speed" if x >= 4 else "low_speed"
)

df["running_type"].value_counts()

running_type
low_speed     679920
high_speed     16240
Name: count, dtype: int64

In [13]:
#Estimate Distance
#Estimate Distance
df = df.sort_values("time")
df["delta_time"] = (
    df["time"]
    .diff()
    .dt.total_seconds()
    .fillna(0)
)
df["distance"] = df["speed"] * df["delta_time"]

In [14]:
distance_summary = (
    df.groupby(
        ["quarter", "running_type"]
    )["distance"]
    .sum()
    .reset_index()
)

distance_summary

,quarter,running_type,distance
0,Q1,high_speed,59.753937
1,Q1,low_speed,1498.910659
2,Q2,high_speed,328.488319
3,Q2,low_speed,1542.384301
4,Q3,high_speed,187.110983
5,Q3,low_speed,1454.947005
6,Q4,high_speed,175.589586
7,Q4,low_speed,1293.691317


Task 2.1 — Distance Covered Per Quarter ✅
* Created Session Timeline
* Converted time to timedelta
* Calculated elapsed seconds
* Split session into:

    * Q1
    * Q2
    * Q3
    * Q4

* Created Running Types
* Low Speed (< 4 m/s)
* High Speed (≥ 4 m/s)

* Calculated
    * Distance covered in each quarter
    * Distance covered by running intensity

Output:

* Q1 High Speed Distance
* Q1 Low Speed Distance
* Q2 High Speed Distance
* Q2 Low Speed Distance
* Q3 High Speed Distance
* Q3 Low Speed Distance
* Q4 High Speed Distance
* Q4 Low Speed Distance

# Task 3  Heart Rate Analysis 

In [15]:
heart_rate_summary = (
    df.groupby("quarter")["heart_rate"]
    .mean()
    .reset_index()
)

heart_rate_summary.head()

,quarter,heart_rate
0,Q1,131.616828
1,Q2,153.961795
2,Q3,153.575697
3,Q4,151.816983


Interpretation
* Q1 is the warm-up / lower intensity phase.
* Q2 and Q3 are the most demanding parts of the session.
* Q4 remains intense but slightly lower than Q2/Q3.

This is exactly the type of feature sports scientists use to measure training intensity.

# Task 4  Average Movement Per Quarter

In [16]:
import numpy as np

df["movement"] = np.sqrt(
    df["accl_x"]**2 +
    df["accl_y"]**2 +
    df["accl_z"]**2
)

In [17]:
imu_summary = (
    df.groupby("quarter")["movement"]
    .mean()
    .reset_index()
)

imu_summary

,quarter,movement
0,Q1,1.206862
1,Q2,1.213131
2,Q3,1.207583
3,Q4,1.203390


What Does This Mean?

we ou calculated:

Movement Magnitude
=
√(accl_x² + accl_y² + accl_z²)

This combines all 3 accelerometer axes into a single movement score.

Interpretation
Q2 has the highest movement intensity.
Q4 has the lowest movement intensity.
The session movement load is fairly consistent across all quarters.

# Task 5 
add injured flag if a player is currently injured
during their training session

In [18]:
#injury.head()

There is no Injury Dataset

In [19]:

injury = pd.read_csv(
    "../Data/subjective/injury/injury.csv"
)

In [20]:
injury.shape

(162, 3)

In [21]:
injury.columns

Index(['player_name', 'type', 'timestamp'], dtype='str')

In [22]:
injury.head()

,player_name,type,timestamp
0,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,"{""right_thigh"":""minor""}",20.03.2020
1,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",20.03.2020
2,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",21.03.2020
3,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",21.03.2020
4,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,"{""right_foot"":""minor""}",22.03.2020


This dataset does NOT contain:

injury_start_date
injury_end_date
return_to_play_date

It only records injury events.

So we cannot accurately determine:

player was injured throughout the entire training session

because we don't know when the injury ended.

In [23]:
#Creating Injury Flags
injury["timestamp"] = pd.to_datetime(
    injury["timestamp"],
    format="%d.%m.%Y"
)

injury["injured"] = 1

injury_flag = injury[
    ["player_name", "timestamp", "injured"]
]

What This Means

If:

Player X
20-03-2020
appears in injury.csv

then:

injured = 1

Otherwise:

injured = 0

In [24]:
injury_flag.head()

,player_name,timestamp,injured
0,TeamA-b58af410-da77-479e-b93c-e03617b9f36d,2020-03-20,1
1,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,2020-03-20,1
2,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,2020-03-21,1
3,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,2020-03-21,1
4,TeamA-5cd7a61b-88b2-46d2-94f8-5a0d4f682d93,2020-03-22,1


SAVE INJURY FLAG

In [ ]:
from pathlib import Path

output_path = Path("../Data/Processed/injury/injury_flag.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

injury_flag.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: ..\Data\Processed\injury\injury_flag.csv


Merging With Training Sessions

In [26]:
session_date = pd.Timestamp("2020-06-01")

In [27]:
%whos DataFrame

Variable             Type         Data/Info
-------------------------------------------
df                   DataFrame    Shape: (696160, 23)
distance_summary     DataFrame    Shape: (8, 3)
heart_rate_summary   DataFrame    Shape: (4, 2)
imu_summary          DataFrame    Shape: (4, 2)
injury               DataFrame    Shape: (162, 4)
injury_flag          DataFrame    Shape: (162, 3)


In [28]:
#Crearting gps_summary
gps_summary = pd.DataFrame([session_summary])

gps_summary

,player_name,avg_speed,max_speed,avg_heart_rate,max_heart_rate,avg_accel_x,avg_accel_y,avg_accel_z,rows
0,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,0.939565,6.455561,147.744218,198,-0.060387,0.84878,0.680267,696160


In [29]:
#adding injury flag to gps_summary
current_player = gps_summary["player_name"].iloc[0]

is_injured = (
    injury_flag["player_name"]
    .eq(current_player)
    .any()
)

gps_summary["injured"] = int(is_injured)

gps_summary

,player_name,avg_speed,max_speed,avg_heart_rate,max_heart_rate,avg_accel_x,avg_accel_y,avg_accel_z,rows,injured
0,TeamA-2d44f941-2f24-4fc2-afa8-611a091f2e93,0.939565,6.455561,147.744218,198,-0.060387,0.84878,0.680267,696160,0
